# NB_LIBRO_03 · Figuras del capítulo 4 (Segmentación y medición clásica)

**Libro:** *AgroVisión: del píxel a la decisión*

**Usa (solo lectura):**

- `PALMA_AFRICANA_YOLO/02_DATOS_PROCESADOS/classification_5class_v1` (452 imágenes)
- `AgroUTMACH_PIX/01_DATASET_PIO/PIO_TRAIN_VAL/data` (imágenes y etiquetas de `train` y `val`)

**Produce en `AGROVISION_LIBRO/`:** las figuras 4.1 a 4.7 en `02_FIGURAS/cap04/`, y en `03_VALORES/` el archivo `valores_cap04.json` y tres tablas CSV.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda entre 10 y 15 minutos; casi todo el tiempo se va en leer desde Drive unas 570 imágenes de pollos. Al terminar, ejecute `NB_LIBRO_00_Empaquetar_resultados_v2` y suba el zip al chat.

## 1. Configuración

In [ ]:
import os
import io
import re
import json
import math
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Carpeta raíz de Drive. Fuera de Colab puede indicarse con la variable AGROVISION_BASE.
BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))

DIR_PALMA = BASE / 'PALMA_AFRICANA_YOLO' / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
DIR_PIO = BASE / 'AgroUTMACH_PIX' / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'

DIR_LIBRO = BASE / 'AGROVISION_LIBRO'
DIR_FIG4 = DIR_LIBRO / '02_FIGURAS' / 'cap04'
DIR_VAL = DIR_LIBRO / '03_VALORES'
for d in (DIR_FIG4, DIR_VAL):
    d.mkdir(parents=True, exist_ok=True)

# --- Parámetros que puede cambiar -------------------------------------------
# Imagen de palma usada como ejemplo. Con None se usa la misma del capítulo 2
# (se lee de 03_VALORES/valores_para_texto.json).
IMG_MADURO = None
# Imagen de pollos usada como ejemplo. Con None se elige la imagen de validación
# cuyo número de aves anotadas está más cerca de la mediana.
IMG_POLLOS = None
# Imágenes de entrenamiento por semana usadas para calibrar el conteo por área.
N_CALIBRA = 20
SEMILLA = 2026
# -----------------------------------------------------------------------------

assert DIR_PALMA.exists(), f'No se encuentra el dataset de palma en: {DIR_PALMA}'
assert (DIR_PIO / 'images' / 'val').exists(), f'No se encuentra el dataset PIO en: {DIR_PIO}'
print('Dataset de palma :', DIR_PALMA)
print('Dataset PIO      :', DIR_PIO)
print('Salida           :', DIR_LIBRO)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)


import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

LADO_CAL = 512        # lado mayor al que se llevan las imágenes antes de medir su calidad
RNG = np.random.default_rng(SEMILLA)
BORDE_BLANCO = [pe.Stroke(linewidth=2.0, foreground='#0b0b0b'), pe.Normal()]


def miles(x):
    # Entero con espacio como separador de miles.
    return f'{x:,.0f}'.replace(',', ' ')


def elemento(k):
    # Elemento estructurante elíptico de k × k píxeles.
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def rotulo_fila(ax, texto):
    ax.set_ylabel(texto, fontsize=8, color=TINTA)

## 2. Inventario del dataset de palma e imagen de ejemplo

In [ ]:
filas_inv = []
for ruta in sorted(DIR_PALMA.rglob('*')):
    if not ruta.is_file() or ruta.suffix.lower() not in EXT_IMG:
        continue
    partes = [p.lower() for p in ruta.relative_to(DIR_PALMA).parts[:-1]]
    clase = next((p for p in partes if p in CLASES), None)
    particion = next((p for p in partes if p in ('train', 'val', 'test')), 'sin_split')
    if clase is None:
        continue
    filas_inv.append({'ruta': str(ruta), 'archivo': ruta.name, 'split': particion, 'clase': clase})
inv = pd.DataFrame(filas_inv)
assert len(inv) > 0, 'No se encontraron imágenes. Revise DIR_PALMA.'
print(inv.groupby(['clase', 'split']).size().unstack(fill_value=0).reindex(CLASES).rename(index=NOMBRE_ES).to_string())
print('Total de imágenes:', len(inv))

# Imagen de ejemplo: la misma del capítulo 2, salvo que se indique otra.
nombre_ej = IMG_MADURO
ruta_val01 = DIR_VAL / 'valores_para_texto.json'
if nombre_ej is None and ruta_val01.exists():
    with open(ruta_val01, encoding='utf-8') as fh:
        nombre_ej = json.load(fh).get('imagenes_ejemplo', {}).get('maduro', {}).get('archivo')
hit = inv[inv['archivo'] == nombre_ej] if nombre_ej else inv.iloc[0:0]
if len(hit) == 0:
    hit = inv[(inv['clase'] == 'ripe') & (inv['split'] == 'train')]
    print('AVISO: no se encontró la imagen de ejemplo del capítulo 2; se usa la primera de la clase maduro.')
fila_ej = hit.iloc[0]
img_ej_full = leer_rgb(fila_ej['ruta'])
img_ej = redimensionar(img_ej_full, LADO_CAL) if max(img_ej_full.shape[:2]) > LADO_CAL else img_ej_full
print('Imagen de ejemplo:', fila_ej['archivo'], img_ej_full.shape[1], '×', img_ej_full.shape[0])
VALORES['imagen_ejemplo_palma'] = {'archivo': fila_ej['archivo'], 'ancho': int(img_ej_full.shape[1]),
                                   'alto': int(img_ej_full.shape[0])}

In [ ]:
# Imágenes de palma reducidas a 512 px de lado mayor, en memoria.
IMG_CAL = {}
for k, fila in enumerate(inv.itertuples(index=False), start=1):
    IMG_CAL[fila.ruta] = leer_rgb(fila.ruta, lado_max=LADO_CAL)
    if k % 50 == 0 or k == len(inv):
        print(f'  {k}/{len(inv)} imágenes leídas')

## 3. Inventario del dataset PIO

Los nombres de archivo codifican el galpón (`C` comercial, `P` prototipo) y la semana de edad (`W1`…`W6`). El número de aves anotadas es el número de líneas válidas del archivo de etiquetas.

In [ ]:
PATRON_PIO = re.compile(r'^([CP])-W(\d+)', re.IGNORECASE)


def listar_pio(particion):
    filas_p = []
    for ruta in sorted((DIR_PIO / 'images' / particion).iterdir()):
        if ruta.suffix.lower() not in EXT_IMG:
            continue
        hallado = PATRON_PIO.match(ruta.name)
        filas_p.append({'ruta': str(ruta), 'archivo': ruta.name, 'particion': particion,
                        'galpon': hallado.group(1).upper() if hallado else '?',
                        'semana': int(hallado.group(2)) if hallado else 0,
                        'etiqueta': str(DIR_PIO / 'labels' / particion / (ruta.stem + '.txt'))})
    return pd.DataFrame(filas_p)


def contar_anotadas(ruta_txt):
    p = Path(ruta_txt)
    if not p.exists():
        return np.nan
    return sum(1 for linea in p.read_text().splitlines() if len(linea.split()) == 5)


pio_val = listar_pio('val')
pio_val['anotadas'] = [contar_anotadas(r) for r in pio_val['etiqueta']]
sin_etq = int(pio_val['anotadas'].isna().sum())
pio_val = pio_val.dropna(subset=['anotadas']).reset_index(drop=True)
pio_val['anotadas'] = pio_val['anotadas'].astype(int)
print(f'Imágenes de validación: {len(pio_val)} (sin etiqueta: {sin_etq})')
print(pio_val.groupby(['semana', 'galpon'])['anotadas'].agg(['count', 'mean', 'min', 'max']).round(0).to_string())

# Muestra de entrenamiento para calibrar el conteo por área (N_CALIBRA por semana).
pio_train = listar_pio('train')
muestra_cal = pd.concat([g.sample(min(N_CALIBRA, len(g)), random_state=SEMILLA)
                         for _, g in pio_train.groupby('semana')]).reset_index(drop=True)
muestra_cal['anotadas'] = [contar_anotadas(r) for r in muestra_cal['etiqueta']]
muestra_cal = muestra_cal.dropna(subset=['anotadas']).reset_index(drop=True)
print(f'\nImágenes de entrenamiento: {len(pio_train)} · muestra de calibración: {len(muestra_cal)}')

VALORES['pio_inventario'] = {
    'n_val': int(len(pio_val)), 'n_train': int(len(pio_train)), 'n_calibracion': int(len(muestra_cal)),
    'aves_por_imagen_val': {'media': round(float(pio_val['anotadas'].mean()), 1), 'mediana': float(pio_val['anotadas'].median()),
                            'min': int(pio_val['anotadas'].min()), 'max': int(pio_val['anotadas'].max())},
    'val_por_semana': {int(s): {'imagenes': int(len(g)), 'aves_media': round(float(g['anotadas'].mean()), 1)}
                       for s, g in pio_val.groupby('semana')},
}

## 5. Umbralización

### Figura 4.1 · Umbral fijo, Otsu y umbral adaptativo en una imagen de aves

In [ ]:
ANCHO_PIO = 960                 # las imágenes de 1920 × 1080 se reducen a la mitad


def leer_pio(ruta):
    im = leer_rgb(ruta)
    esc = ANCHO_PIO / im.shape[1]
    return cv2.resize(im, (ANCHO_PIO, round(im.shape[0] * esc)), interpolation=cv2.INTER_AREA)


if IMG_POLLOS:
    fila_pio = pio_val[pio_val['archivo'] == IMG_POLLOS].iloc[0]
else:
    fila_pio = pio_val.iloc[(pio_val['anotadas'] - pio_val['anotadas'].median()).abs().argsort().iloc[0]]
img_pio = leer_pio(fila_pio['ruta'])
gris_pio = cv2.GaussianBlur(cv2.cvtColor(img_pio, cv2.COLOR_RGB2GRAY), (5, 5), 0)
print('Imagen de ejemplo:', fila_pio['archivo'], '· aves anotadas:', fila_pio['anotadas'])

T_FIJO = 127
t_otsu, bin_otsu = cv2.threshold(gris_pio, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, bin_fijo = cv2.threshold(gris_pio, T_FIJO, 255, cv2.THRESH_BINARY)
bin_adapt = cv2.adaptiveThreshold(gris_pio, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 51, -8)
gris_osc = np.clip(gris_pio.astype(np.float32) * 0.6, 0, 255).astype(np.uint8)
_, bin_fijo_osc = cv2.threshold(gris_osc, T_FIJO, 255, cv2.THRESH_BINARY)
t_otsu_osc, bin_otsu_osc = cv2.threshold(gris_osc, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

prop_p = img_pio.shape[0] / img_pio.shape[1]
fig, axs = plt.subplots(2, 3, figsize=(ANCHO_IN, ANCHO_IN * prop_p / 3 * 2 + 0.6), constrained_layout=True)
axs[0, 0].imshow(gris_pio, cmap='gray', vmin=0, vmax=255); axs[0, 0].set_title('(a) Imagen en escala de grises')
hist_p = np.bincount(gris_pio.ravel(), minlength=256) / gris_pio.size
axh = axs[0, 1]
axh.fill_between(np.arange(256), hist_p, step='mid', color=GRIS_FONDO, linewidth=0)
axh.axvline(T_FIJO, color=TINTA, linewidth=0.9, linestyle='--', label=f'Umbral fijo = {T_FIJO}')
axh.axvline(t_otsu, color=ACENTO, linewidth=1.3, label=f'Umbral de Otsu = {int(t_otsu)}')
axh.set_xlim(0, 255); axh.set_ylim(0, hist_p.max() * 1.75); axh.set_xlabel('Nivel de gris'); axh.set_ylabel('Proporción de píxeles')
axh.set_title('(b) Histograma y umbrales'); axh.legend(loc='upper center', fontsize=6.3, handlelength=1.6, ncol=1, frameon=True, facecolor='white',
           edgecolor='none', framealpha=1.0, borderpad=0.2)
eje_con_coma(axh.yaxis, 3); axh.set_box_aspect(prop_p)
axs[0, 2].imshow(bin_fijo, cmap='gray'); axs[0, 2].set_title(f'(c) Umbral fijo, T = {T_FIJO}')
axs[1, 0].imshow(bin_otsu, cmap='gray'); axs[1, 0].set_title(f'(d) Otsu, T = {int(t_otsu)}')
axs[1, 1].imshow(bin_adapt, cmap='gray'); axs[1, 1].set_title('(e) Adaptativo, vecindad de 51 px')
axs[1, 2].imshow(bin_fijo_osc, cmap='gray'); axs[1, 2].set_title(f'(f) T = {T_FIJO}, imagen oscurecida')
for ax in (axs[0, 0], axs[0, 2], axs[1, 0], axs[1, 1], axs[1, 2]):
    sin_ejes(ax)
guardar(fig, DIR_FIG4 / 'Fig_4_1_umbralizacion.png')
plt.show()
VALORES['figura_4_1'] = {
    'imagen': fila_pio['archivo'], 'aves_anotadas': int(fila_pio['anotadas']),
    'umbral_otsu': int(t_otsu), 'umbral_otsu_imagen_oscurecida': int(t_otsu_osc),
    'pct_primer_plano': {'fijo_127': round(float((bin_fijo > 0).mean() * 100), 1),
                         'otsu': round(float((bin_otsu > 0).mean() * 100), 1),
                         'adaptativo': round(float((bin_adapt > 0).mean() * 100), 1),
                         'fijo_127_oscurecida': round(float((bin_fijo_osc > 0).mean() * 100), 1),
                         'otsu_oscurecida': round(float((bin_otsu_osc > 0).mean() * 100), 1)},
}
print(json.dumps(VALORES['figura_4_1'], ensure_ascii=False, indent=1))

## 6. Segmentación del racimo por color

El fondo se estima con la mediana CIELAB de un anillo en el borde de la imagen. Cada píxel recibe su distancia de color ΔE a ese fondo; el umbral de Otsu sobre ese mapa separa racimo y fondo. La máscara se limpia con apertura y cierre, se conserva el componente mayor y se rellenan sus huecos.

In [ ]:
ANCHO_ANILLO = 0.06             # ancho del anillo de borde, como fracción del lado menor
DE_MAX = 60.0                   # distancia de color que se lleva a 255 antes de aplicar Otsu
K_APERTURA, K_CIERRE = 7, 21    # tamaños de los elementos estructurantes, en píxeles


def segmentar_racimo(rgb):
    h, w = rgb.shape[:2]
    b = max(4, round(ANCHO_ANILLO * min(h, w)))
    anillo = np.ones((h, w), bool); anillo[b:h - b, b:w - b] = False
    lab = rgb2lab(rgb)
    fondo = np.median(lab[anillo], axis=0)
    dE = cv2.GaussianBlur(np.linalg.norm(lab - fondo, axis=2).astype(np.float32), (0, 0), 2)
    d8 = np.clip(dE * 255.0 / DE_MAX, 0, 255).astype(np.uint8)
    t8, inicial = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    abierta = cv2.morphologyEx(inicial, cv2.MORPH_OPEN, elemento(K_APERTURA))
    cerrada = cv2.morphologyEx(abierta, cv2.MORPH_CLOSE, elemento(K_CIERRE))
    n_comp, etiquetas, estad, _ = cv2.connectedComponentsWithStats(cerrada, connectivity=8)
    final = np.zeros((h, w), np.uint8)
    contorno = None
    if n_comp > 1:
        mayor = 1 + int(np.argmax(estad[1:, cv2.CC_STAT_AREA]))
        solo = np.where(etiquetas == mayor, 255, 0).astype(np.uint8)
        cs, _ = cv2.findContours(solo, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contorno = max(cs, key=cv2.contourArea)
        cv2.drawContours(final, [contorno], -1, 255, thickness=cv2.FILLED)      # rellena los huecos
    return {'lab': lab, 'fondo': fondo, 'dE': dE, 'umbral_dE': t8 * DE_MAX / 255.0, 'inicial': inicial,
            'abierta': abierta, 'cerrada': cerrada, 'final': final, 'contorno': contorno, 'anillo': anillo, 'b': b,
            'componentes': int(n_comp - 1)}


def medir(seg):
    # Medidas geométricas y de color de la máscara final.
    h, w = seg['final'].shape
    c = seg['contorno']
    if c is None or cv2.contourArea(c) < 10:
        return None
    area = float(cv2.contourArea(c)); perim = float(cv2.arcLength(c, True))
    mom = cv2.moments(c)
    x, y, bw, bh = cv2.boundingRect(c)
    casco = float(cv2.contourArea(cv2.convexHull(c)))
    dentro = seg['final'] > 0
    lab_m = seg['lab'][dentro].mean(axis=0)
    return {'area_px': area, 'area_pct': area / (h * w) * 100, 'perimetro_px': perim,
            'circularidad': 4 * math.pi * area / perim ** 2 if perim > 0 else 0.0,
            'diametro_equivalente_px': math.sqrt(4 * area / math.pi),
            'solidez': area / casco if casco > 0 else 0.0,
            'cx': mom['m10'] / mom['m00'], 'cy': mom['m01'] / mom['m00'],
            'caja_x': x, 'caja_y': y, 'caja_ancho': bw, 'caja_alto': bh,
            'relacion_aspecto': max(bw, bh) / max(1, min(bw, bh)),
            'pct_anillo_cubierto': float(dentro[seg['anillo']].mean() * 100),
            'L_mascara': float(lab_m[0]), 'a_mascara': float(lab_m[1]), 'b_mascara': float(lab_m[2])}


def plausible(m):
    # Sin máscaras de referencia no puede medirse el acierto; se usa un criterio de plausibilidad:
    # el racimo ocupa entre el 5 % y el 75 % de la imagen y apenas toca el borde.
    return m is not None and 5 <= m['area_pct'] <= 75 and m['pct_anillo_cubierto'] < 15


def dibujar_contorno(ax, contorno, **kw):
    pts = contorno[:, 0, :]
    pts = np.vstack([pts, pts[:1]])
    linea, = ax.plot(pts[:, 0], pts[:, 1], color='white', linewidth=kw.get('lw', 1.3), solid_joinstyle='round')
    linea.set_path_effects(BORDE_BLANCO)


seg_ej = segmentar_racimo(img_ej)
med_ej = medir(seg_ej)
print('Fondo estimado (L*, a*, b*):', np.round(seg_ej['fondo'], 1), '| umbral ΔE:', round(seg_ej['umbral_dE'], 1))
print({k: round(v, 2) for k, v in med_ej.items()})

### Figura 4.2 · Segmentación por distancia de color al fondo

In [ ]:
h_s, w_s = img_ej.shape[:2]
prop_s = h_s / w_s
fig, axs = plt.subplots(1, 4, figsize=(ANCHO_IN, ANCHO_IN * prop_s / 4 + 0.42), constrained_layout=True)
axs[0].imshow(img_ej)
b_s = seg_ej['b']
axs[0].add_patch(Rectangle((b_s, b_s), w_s - 2 * b_s, h_s - 2 * b_s, fill=False, edgecolor='white', linewidth=1.0,
                           linestyle=(0, (4, 2)), path_effects=[pe.Stroke(linewidth=2.2, foreground=TINTA), pe.Normal()]))
axs[0].set_title('(a) Imagen y anillo', fontsize=7.5)
axs[1].imshow(seg_ej['dE'], cmap='gray', vmin=0, vmax=DE_MAX); axs[1].set_title('(b) Distancia de color ΔE', fontsize=7.5)
for ax in (axs[0], axs[1], axs[3]):
    sin_ejes(ax)
dE_plano = seg_ej['dE'].ravel()
axs[2].hist(dE_plano, bins=np.linspace(0, DE_MAX, 61), color=GRIS_FONDO, edgecolor='none', density=True)
axs[2].axvline(seg_ej['umbral_dE'], color=ACENTO, linewidth=1.3)
axs[2].text(seg_ej['umbral_dE'] + 1.5, axs[2].get_ylim()[1] * 0.93, f"Otsu\nΔE = {coma(seg_ej['umbral_dE'], 1)}",
            fontsize=6.3, color=ACENTO, va='top', linespacing=1.1)
axs[2].set_xlim(0, DE_MAX); axs[2].set_xlabel('ΔE respecto al fondo'); axs[2].set_yticks([])
axs[2].set_title('(c) Histograma de ΔE', fontsize=7.5); axs[2].set_box_aspect(prop_s)
axs[3].imshow(seg_ej['inicial'], cmap='gray'); axs[3].set_title('(d) Máscara inicial', fontsize=7.5)
guardar(fig, DIR_FIG4 / 'Fig_4_2_segmentacion_color.png')
plt.show()

### Figura 4.3 · Operaciones morfológicas sobre la máscara

In [ ]:
K_DEMO = 9
pasos_morf = [
    ('Máscara inicial', seg_ej['inicial']),
    (f'Erosión ({K_DEMO} × {K_DEMO})', cv2.erode(seg_ej['inicial'], elemento(K_DEMO))),
    (f'Dilatación ({K_DEMO} × {K_DEMO})', cv2.dilate(seg_ej['inicial'], elemento(K_DEMO))),
    (f'Apertura ({K_APERTURA} × {K_APERTURA})', seg_ej['abierta']),
    (f'Cierre de la apertura ({K_CIERRE} × {K_CIERRE})', seg_ej['cerrada']),
    ('Componente mayor, sin huecos', seg_ej['final']),
]
fig, axs = plt.subplots(2, 3, figsize=(ANCHO_IN, ANCHO_IN * prop_s / 3 * 2 + 0.55), constrained_layout=True)
info_morf = {}
for ax, letra, (nombre, v) in zip(axs.ravel(), 'abcdef', pasos_morf):
    ax.imshow(v, cmap='gray', vmin=0, vmax=255); sin_ejes(ax)
    for borde_ax in ax.spines.values():
        borde_ax.set_visible(True); borde_ax.set_color(GRIS_FONDO); borde_ax.set_linewidth(0.5)
    n_c = cv2.connectedComponents(v)[0] - 1
    info_morf[nombre] = {'pct_blanco': round(float((v > 0).mean() * 100), 1), 'componentes': int(n_c)}
    ax.set_title(f'({letra}) {nombre}', fontsize=7.5)
guardar(fig, DIR_FIG4 / 'Fig_4_3_morfologia.png')
plt.show()
VALORES['figura_4_2_4_3'] = {
    'fondo_Lab': [round(float(v), 1) for v in seg_ej['fondo']],
    'umbral_dE': round(float(seg_ej['umbral_dE']), 1),
    'pasos': info_morf,
    'medidas_ejemplo': {k: round(float(v), 3 if k in ('circularidad', 'solidez') else 1) for k, v in med_ej.items()},
    'dimension_imagen': f'{w_s} × {h_s}',
}
print(json.dumps(info_morf, ensure_ascii=False, indent=1))

## 7. Segmentación y medidas de las 452 imágenes

Se segmenta cada imagen con los mismos parámetros, sin ajuste por imagen. Con el color medio dentro de la máscara se repite la línea de base del capítulo 2 (centroide de color más cercano) para comprobar si aislar el racimo mejora la clasificación por color.

In [ ]:
filas_seg, SEGS = [], {}
for k, fila in enumerate(inv.itertuples(index=False), start=1):
    s = segmentar_racimo(IMG_CAL[fila.ruta])
    m = medir(s)
    SEGS[fila.ruta] = s['contorno']
    reg = {'archivo': fila.archivo, 'ruta': fila.ruta, 'split': fila.split, 'clase': fila.clase,
           'umbral_dE': s['umbral_dE'], 'componentes': s['componentes'], 'plausible': plausible(m)}
    reg.update(m or {})
    filas_seg.append(reg)
    if k % 50 == 0 or k == len(inv):
        print(f'  {k}/{len(inv)} imágenes segmentadas')
seg = pd.DataFrame(filas_seg)
seg.drop(columns=['ruta']).round(3).to_csv(DIR_VAL / 'segmentacion_por_imagen.csv', index=False)

res_seg = seg.groupby('clase').agg(imagenes=('archivo', 'size'), plausibles=('plausible', 'sum'),
                                   area_pct=('area_pct', 'median'), circularidad=('circularidad', 'median'),
                                   solidez=('solidez', 'median'), relacion_aspecto=('relacion_aspecto', 'median')).reindex(CLASES)
res_seg['pct_plausibles'] = res_seg['plausibles'] / res_seg['imagenes'] * 100
print(res_seg.rename(index=NOMBRE_ES).round(2).to_string())
print(f"\nMáscaras plausibles: {int(seg['plausible'].sum())} de {len(seg)} ({seg['plausible'].mean() * 100:.1f} %)")
no_pl = seg[~seg['plausible']]
motivo_seg = {'sin_mascara': int(no_pl['area_pct'].isna().sum()),
              'area_menor_5pct': int((no_pl['area_pct'] < 5).sum()),
              'area_mayor_75pct': int((no_pl['area_pct'] > 75).sum()),
              'toca_el_borde': int(((no_pl['pct_anillo_cubierto'] >= 15) & no_pl['area_pct'].between(5, 75)).sum())}
print('Motivos de descarte:', motivo_seg)


def linea_base(tabla, cols):
    ent = tabla[tabla['split'] == 'train'].dropna(subset=cols); pru = tabla[tabla['split'] == 'test'].dropna(subset=cols)
    if len(ent) == 0 or len(pru) == 0:
        return None
    cen = ent.groupby('clase')[cols].mean().reindex(CLASES).dropna()
    dist = np.linalg.norm(pru[cols].to_numpy()[:, None, :] - cen.to_numpy()[None, :, :], axis=2)
    pred = np.array(cen.index)[dist.argmin(axis=1)]
    real = pru['clase'].to_numpy()
    f1s = {}
    for c in cen.index:
        vp = int(((real == c) & (pred == c)).sum()); fp = int(((real != c) & (pred == c)).sum())
        fn = int(((real == c) & (pred != c)).sum())
        f1s[NOMBRE_ES[c]] = 0.0 if vp == 0 else round(2 * vp / (2 * vp + fp + fn), 3)
    return {'n_prueba': int(len(pru)), 'exactitud': round(float((real == pred).mean()), 3),
            'f1_macro': round(float(np.mean(list(f1s.values()))), 3), 'f1_por_clase': f1s}


COLS_LAB = ['L_mascara', 'a_mascara', 'b_mascara']
lb_mascara = linea_base(seg, COLS_LAB)
lb_plausibles = linea_base(seg[seg['plausible']], COLS_LAB)
lb_color_forma = None
if lb_mascara:
    # Color y forma juntos, con cada variable estandarizada en el conjunto de entrenamiento.
    cols_cf = COLS_LAB + ['circularidad', 'solidez', 'area_pct']
    ent_cf = seg[seg['split'] == 'train'].dropna(subset=cols_cf)
    z = seg.dropna(subset=cols_cf).copy()
    z[cols_cf] = (z[cols_cf] - ent_cf[cols_cf].mean()) / ent_cf[cols_cf].std()
    lb_color_forma = linea_base(z, cols_cf)
lb_cap2 = None
if ruta_val01.exists():
    with open(ruta_val01, encoding='utf-8') as fh:
        lb_cap2 = json.load(fh).get('color_por_clase', {}).get('linea_base_color')
print('\nLínea de base, región central (cap. 2):', lb_cap2 and {k: lb_cap2[k] for k in ('exactitud', 'f1_macro')})
print('Línea de base, color dentro de la máscara:', lb_mascara)
print('Solo máscaras plausibles:', lb_plausibles)
print('Color + forma:', lb_color_forma)

color_masc = seg.groupby('clase')[COLS_LAB].agg(['mean', 'std']).reindex(CLASES)
VALORES['segmentacion_palma'] = {
    'parametros': {'lado_mayor_px': LADO_CAL, 'anillo': ANCHO_ANILLO, 'apertura': K_APERTURA, 'cierre': K_CIERRE},
    'n_imagenes': int(len(seg)), 'plausibles': int(seg['plausible'].sum()),
    'pct_plausibles': round(float(seg['plausible'].mean() * 100), 1),
    'motivos_descarte': motivo_seg,
    'por_clase': {NOMBRE_ES[c]: {k: round(float(v), 2) for k, v in res_seg.loc[c].items()} for c in CLASES},
    'umbral_dE_mediana': round(float(seg['umbral_dE'].median()), 1),
    'color_en_mascara': {NOMBRE_ES[c]: {q[0]: f"{color_masc.loc[c, (q, 'mean')]:.1f} ± {color_masc.loc[c, (q, 'std')]:.1f}"
                                         for q in COLS_LAB} for c in CLASES},
    'linea_base_region_central_cap2': lb_cap2,
    'linea_base_color_en_mascara': lb_mascara,
    'linea_base_color_en_mascara_solo_plausibles': lb_plausibles,
    'linea_base_color_y_forma': lb_color_forma,
}

### Figura 4.4 · Contorno, caja y centroide en las cinco clases

In [ ]:
def panel_contorno(ax, reg):
    im = IMG_CAL[reg['ruta']]
    ax.imshow(im); sin_ejes(ax); ax.set_box_aspect(0.78)
    c = SEGS[reg['ruta']]
    if c is None or pd.isna(reg.get('area_pct', np.nan)):
        ax.set_title('Sin máscara', fontsize=6.5)
        return
    dibujar_contorno(ax, c, lw=0.9)
    ax.add_patch(Rectangle((reg['caja_x'], reg['caja_y']), reg['caja_ancho'], reg['caja_alto'], fill=False,
                           edgecolor='white', linewidth=0.8, linestyle=(0, (3, 2)),
                           path_effects=[pe.Stroke(linewidth=1.8, foreground=TINTA), pe.Normal()]))
    ax.plot(reg['cx'], reg['cy'], marker='+', markersize=7, markeredgewidth=1.4, color='white',
            path_effects=[pe.Stroke(linewidth=2.8, foreground=TINTA), pe.Normal()])
    ax.set_title(f"Área {coma(reg['area_pct'], 0)} % · circ. {coma(reg['circularidad'], 2)}", fontsize=6.5)


fig, axs = plt.subplots(2, 5, figsize=(ANCHO_IN, ANCHO_IN * 0.78 / 5 * 2 + 0.85), constrained_layout=True)
info_44 = {}
for k, clase in enumerate(CLASES):
    sub_c = seg[seg['clase'] == clase]
    buenas = sub_c[sub_c['plausible']]
    malas = sub_c[~sub_c['plausible']]
    if len(buenas):
        tipica = buenas.iloc[(buenas['solidez'] - buenas['solidez'].median()).abs().argsort().iloc[0]]
    else:
        tipica = sub_c.iloc[0]
    if len(malas):
        dudosa = malas.sort_values('pct_anillo_cubierto', ascending=False, na_position='first').iloc[0]
    else:
        dudosa = buenas.sort_values('solidez').iloc[0]
    panel_contorno(axs[0, k], tipica); panel_contorno(axs[1, k], dudosa)
    axs[0, k].set_title(f"{NOMBRE_ES[clase]}\n" + axs[0, k].get_title(), fontsize=6.8, linespacing=1.25)
    info_44[NOMBRE_ES[clase]] = {'tipica': tipica['archivo'], 'dudosa': dudosa['archivo'],
                                 'dudosa_es_no_plausible': bool(len(malas) > 0)}
rotulo_fila(axs[0, 0], 'Plausible'); rotulo_fila(axs[1, 0], 'Fallida o dudosa')
guardar(fig, DIR_FIG4 / 'Fig_4_4_contornos_clases.png')
plt.show()
VALORES['figura_4_4'] = info_44

## 8. Conteo de aves con visión clásica

Tres estimadores sobre la misma máscara (Otsu sobre el gris, descarte de píxeles muy saturados y apertura):

- **A · componentes**: número de componentes conexos;
- **B · área de referencia**: cada componente aporta su área dividida entre el área mediana de los componentes de la imagen;
- **C · área calibrada**: área total de primer plano dividida entre el área media por ave de esa semana de edad, calibrada con imágenes de entrenamiento y sus anotaciones.

In [ ]:
S_MAX = 90            # saturación máxima (0–255) admitida para un píxel de ave: descarta comederos amarillos
AREA_MIN = 20         # área mínima de un componente, en píxeles (imagen de 960 px de ancho)
K_AVES = 5


def segmentar_aves(rgb):
    gris = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    t, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    sat = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 1]
    filtrada = binaria.copy(); filtrada[sat > S_MAX] = 0
    abierta = cv2.morphologyEx(filtrada, cv2.MORPH_OPEN, elemento(K_AVES))
    n, etiquetas, estad, _ = cv2.connectedComponentsWithStats(abierta, connectivity=8)
    areas = estad[1:, cv2.CC_STAT_AREA].astype(np.float64)
    validos = np.where(areas >= AREA_MIN)[0] + 1
    return {'umbral': float(t), 'binaria': binaria, 'filtrada': filtrada, 'abierta': abierta,
            'pct_quitado_saturacion': float(((binaria > 0) & (sat > S_MAX)).sum() / max(1, (binaria > 0).sum()) * 100),
            'etiquetas': etiquetas, 'validos': validos, 'areas': areas[validos - 1]}


def estimar(sa):
    a = sa['areas']
    if len(a) == 0:
        return {'n_componentes': 0, 'n_area_ref': 0, 'area_primer_plano': 0.0, 'area_ref': np.nan, 'por_componente': a}
    ref = float(np.median(a))
    por_comp = np.maximum(1, np.round(a / ref))
    return {'n_componentes': int(len(a)), 'n_area_ref': int(por_comp.sum()), 'area_primer_plano': float(a.sum()),
            'area_ref': ref, 'por_componente': por_comp}


# --- Calibración del estimador C con la muestra de entrenamiento ---
filas_calib = []
for k, fila in enumerate(muestra_cal.itertuples(index=False), start=1):
    e = estimar(segmentar_aves(leer_pio(fila.ruta)))
    filas_calib.append({'semana': fila.semana, 'anotadas': fila.anotadas, 'area_primer_plano': e['area_primer_plano']})
    if k % 30 == 0 or k == len(muestra_cal):
        print(f'  calibración: {k}/{len(muestra_cal)}')
calib = pd.DataFrame(filas_calib)
area_por_ave = (calib.groupby('semana')['area_primer_plano'].sum() / calib.groupby('semana')['anotadas'].sum())
area_global = float(calib['area_primer_plano'].sum() / calib['anotadas'].sum())
print('Área de primer plano por ave anotada (px², imagen de 960 px de ancho):')
print(area_por_ave.round(1).to_string())

# --- Evaluación en validación ---
filas_cnt = []
for k, fila in enumerate(pio_val.itertuples(index=False), start=1):
    sa = segmentar_aves(leer_pio(fila.ruta))
    e = estimar(sa)
    a_ref = float(area_por_ave.get(fila.semana, area_global))
    filas_cnt.append({'archivo': fila.archivo, 'ruta': fila.ruta, 'galpon': fila.galpon, 'semana': fila.semana,
                      'anotadas': fila.anotadas, 'umbral_otsu': sa['umbral'],
                      'pct_quitado_saturacion': sa['pct_quitado_saturacion'],
                      'pct_primer_plano': e['area_primer_plano'] / sa['abierta'].size * 100,
                      'A_componentes': e['n_componentes'], 'B_area_ref': e['n_area_ref'],
                      'C_area_calibrada': int(round(e['area_primer_plano'] / a_ref)) if a_ref > 0 else 0})
    if k % 50 == 0 or k == len(pio_val):
        print(f'  validación: {k}/{len(pio_val)}')
cnt = pd.DataFrame(filas_cnt)
cnt.drop(columns=['ruta']).round(2).to_csv(DIR_VAL / 'conteo_clasico_por_imagen.csv', index=False)

METODOS = {'A_componentes': 'A · Componentes', 'B_area_ref': 'B · Área de referencia', 'C_area_calibrada': 'C · Área calibrada'}


def errores(tabla, col):
    d = tabla[col] - tabla['anotadas']
    return {'MAE': round(float(d.abs().mean()), 1), 'sesgo': round(float(d.mean()), 1),
            'MAPE_pct': round(float((d.abs() / tabla['anotadas'].clip(lower=1)).mean() * 100), 1),
            'RMSE': round(float(np.sqrt((d ** 2).mean())), 1),
            'r': round(float(np.corrcoef(tabla[col], tabla['anotadas'])[0, 1]), 3) if len(tabla) > 2 and tabla[col].std() > 0 else None}


err_global = {METODOS[c]: errores(cnt, c) for c in METODOS}
err_semana = {int(s): {METODOS[c]: errores(g, c) for c in METODOS} | {'imagenes': int(len(g)), 'aves_media': round(float(g['anotadas'].mean()), 1)}
              for s, g in cnt.groupby('semana')}
err_galpon = {str(s): {METODOS[c]: errores(g, c) for c in METODOS} | {'imagenes': int(len(g))} for s, g in cnt.groupby('galpon')}
print('\nError global en validación:')
print(pd.DataFrame(err_global).T.to_string())
print('\nMAE por semana:')
print(pd.DataFrame({s: {m: v[m]['MAE'] for m in METODOS.values()} for s, v in err_semana.items()}).T.to_string())
tabla_sem = pd.DataFrame([{'semana': s, 'imagenes': v['imagenes'], 'aves_media': v['aves_media'],
                           **{f'{m} MAE': v[m]['MAE'] for m in METODOS.values()},
                           **{f'{m} sesgo': v[m]['sesgo'] for m in METODOS.values()}} for s, v in err_semana.items()])
tabla_sem.to_csv(DIR_VAL / 'tabla_4_conteo_clasico_por_semana.csv', index=False)

VALORES['conteo_clasico'] = {
    'parametros': {'ancho_px': ANCHO_PIO, 'saturacion_max': S_MAX, 'area_min_px': AREA_MIN, 'apertura': K_AVES,
                   'imagenes_calibracion_por_semana': N_CALIBRA},
    'area_por_ave_px2_por_semana': {int(s): round(float(v), 1) for s, v in area_por_ave.items()},
    'n_val': int(len(cnt)), 'aves_media_val': round(float(cnt['anotadas'].mean()), 1),
    'error_global': err_global, 'error_por_semana': err_semana, 'error_por_galpon': err_galpon,
    'umbral_otsu': {'mediana': float(cnt['umbral_otsu'].median()), 'min': float(cnt['umbral_otsu'].min()),
                    'max': float(cnt['umbral_otsu'].max())},
    'pct_primer_plano_por_semana': {int(k): round(float(v), 1) for k, v in cnt.groupby('semana')['pct_primer_plano'].mean().items()},
    'pct_quitado_saturacion_por_semana': {int(k): round(float(v), 1) for k, v in cnt.groupby('semana')['pct_quitado_saturacion'].mean().items()},
    'componentes_por_ave_por_semana': {int(k): round(float(g['A_componentes'].sum() / g['anotadas'].sum()), 2) for k, g in cnt.groupby('semana')},
    'referencia_YOLO11s_MAE_NB08': 13.9, 'referencia_FasterRCNN_MAE_NB08': 41.6,
}

### Figura 4.5 · Los pasos del conteo clásico

In [ ]:
sa_ej = segmentar_aves(img_pio)
e_ej = estimar(sa_ej)
RAMPA = ['#a9cdf5', '#2a78d6', '#0d366b']           # 1 ave · 2 o 3 aves · 4 o más
fondo_gris = cv2.cvtColor(img_pio, cv2.COLOR_RGB2GRAY).astype(np.float32) * 0.35 + 150
capa = np.repeat(np.clip(fondo_gris, 0, 255)[:, :, None], 3, axis=2).astype(np.uint8)
tabla_color = np.zeros((sa_ej['etiquetas'].max() + 1, 3), np.uint8)
usados = np.zeros(sa_ej['etiquetas'].max() + 1, bool)
for etq_c, n_c in zip(sa_ej['validos'], e_ej['por_componente']):
    tono = RAMPA[0] if n_c <= 1 else RAMPA[1] if n_c <= 3 else RAMPA[2]
    tabla_color[etq_c] = [int(tono[i:i + 2], 16) for i in (1, 3, 5)]
    usados[etq_c] = True
mascara_val = usados[sa_ej['etiquetas']]
capa[mascara_val] = tabla_color[sa_ej['etiquetas']][mascara_val]

fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * prop_p + 0.75), constrained_layout=True)
axs[0, 0].imshow(img_pio); axs[0, 0].set_title(f"(a) Imagen original · {fila_pio['anotadas']} aves anotadas")
axs[0, 1].imshow(sa_ej['binaria'], cmap='gray'); axs[0, 1].set_title(f"(b) Umbral de Otsu, T = {int(sa_ej['umbral'])}")
axs[1, 0].imshow(sa_ej['abierta'], cmap='gray'); axs[1, 0].set_title('(c) Sin píxeles saturados y tras la apertura')
axs[1, 1].imshow(capa); axs[1, 1].set_title(f"(d) {e_ej['n_componentes']} componentes · {e_ej['n_area_ref']} aves estimadas por área")
axs[1, 1].legend(handles=[Patch(facecolor=RAMPA[0], label='1 ave'), Patch(facecolor=RAMPA[1], label='2 o 3 aves'),
                          Patch(facecolor=RAMPA[2], label='4 o más')],
                 loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False, fontsize=6.5, handlelength=1.2)
for ax in axs.ravel():
    sin_ejes(ax)
guardar(fig, DIR_FIG4 / 'Fig_4_5_conteo_clasico_pasos.png')
plt.show()
fila_cnt_ej = cnt[cnt['archivo'] == fila_pio['archivo']].iloc[0]
VALORES['figura_4_5'] = {
    'imagen': fila_pio['archivo'], 'semana': int(fila_pio['semana']), 'galpon': fila_pio['galpon'],
    'aves_anotadas': int(fila_pio['anotadas']), 'umbral_otsu': int(sa_ej['umbral']),
    'componentes': int(e_ej['n_componentes']), 'estimadas_area_ref': int(e_ej['n_area_ref']),
    'estimadas_area_calibrada': int(fila_cnt_ej['C_area_calibrada']),
    'area_referencia_px': round(float(e_ej['area_ref']), 1),
    'componentes_1_ave': int((e_ej['por_componente'] <= 1).sum()),
    'componentes_2_3_aves': int(((e_ej['por_componente'] > 1) & (e_ej['por_componente'] <= 3)).sum()),
    'componentes_4_o_mas': int((e_ej['por_componente'] > 3).sum()),
    'componente_mayor_aves': int(e_ej['por_componente'].max()),
}
print(json.dumps(VALORES['figura_4_5'], ensure_ascii=False, indent=1))

### Figura 4.6 · Conteo clásico frente a aves anotadas, por semana de edad

In [ ]:
semanas = sorted(cnt['semana'].unique())
n_col = 3 if len(semanas) > 2 else len(semanas)
n_fil = math.ceil(len(semanas) / n_col)
fig, axs = plt.subplots(n_fil, n_col, figsize=(ANCHO_IN, ANCHO_IN / n_col * n_fil * 0.98 + 0.5),
                        constrained_layout=True, squeeze=False)
lim_sup = float(max(cnt['anotadas'].max(), cnt['C_area_calibrada'].quantile(0.99), cnt['A_componentes'].quantile(0.99))) * 1.05
for ax, s in zip(axs.ravel(), semanas):
    g = cnt[cnt['semana'] == s]
    ax.plot([0, lim_sup], [0, lim_sup], color=TINTA_2, linewidth=0.7, linestyle='--', zorder=1)
    ax.scatter(g['anotadas'], g['A_componentes'], s=9, facecolors='none', edgecolors='#8a8984', linewidths=0.6, zorder=2)
    ax.scatter(g['anotadas'], g['C_area_calibrada'], s=9, color=ACENTO, linewidths=0, zorder=3)
    ax.set_xlim(0, lim_sup); ax.set_ylim(0, lim_sup); ax.set_aspect('equal')
    ax.grid(color=REJILLA, linewidth=0.5); ax.set_axisbelow(True)
    titulo_s = f'Semana {s}' if s else 'Semana sin identificar'
    ax.set_title(f'{titulo_s} · n = {len(g)}', loc='left')
    mae_a = float((g['A_componentes'] - g['anotadas']).abs().mean()); mae_c = float((g['C_area_calibrada'] - g['anotadas']).abs().mean())
    ax.text(0.04, 0.96, f'EAM A = {coma(mae_a, 1)}\nEAM C = {coma(mae_c, 1)}', transform=ax.transAxes, va='top', ha='left',
            fontsize=6.3, linespacing=1.3, bbox=dict(facecolor='white', edgecolor='none', pad=1.5, alpha=0.9))
for ax in axs.ravel()[len(semanas):]:
    ax.set_visible(False)
for ax in axs[-1, :]:
    ax.set_xlabel('Aves anotadas')
for ax in axs[:, 0]:
    ax.set_ylabel('Aves estimadas')
fig.legend(handles=[Line2D([], [], marker='o', linestyle='', markersize=4, markerfacecolor='none', markeredgecolor='#8a8984',
                           label='A · Número de componentes'),
                    Line2D([], [], marker='o', linestyle='', markersize=4, color=ACENTO, label='C · Área calibrada por semana'),
                    Line2D([], [], color=TINTA_2, linewidth=0.7, linestyle='--', label='Estimación perfecta')],
           loc='outside lower center', ncol=3, frameon=False, fontsize=6.8)
guardar(fig, DIR_FIG4 / 'Fig_4_6_conteo_por_semana.png')
plt.show()

### Figura 4.7 · Cuatro casos del estimador C

In [ ]:
cnt['dif_C'] = cnt['C_area_calibrada'] - cnt['anotadas']
orden_err = cnt.assign(abs_C=cnt['dif_C'].abs()).sort_values('abs_C').reset_index(drop=True)
casos = [('Menor error', orden_err.iloc[0]), ('Error mediano', orden_err.iloc[len(orden_err) // 2]),
         ('Mayor subestimación', cnt.sort_values('dif_C').iloc[0]), ('Mayor sobreestimación', cnt.sort_values('dif_C').iloc[-1])]
fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * prop_p + 0.95), constrained_layout=True)
info_47 = {}
for ax, letra, (nombre, reg) in zip(axs.ravel(), 'abcd', casos):
    im = leer_pio(reg['ruta'])
    sa = segmentar_aves(im)
    ax.imshow(im); sin_ejes(ax)
    mascara_c = np.isin(sa['etiquetas'], sa['validos']).astype(np.uint8)
    cs, _ = cv2.findContours(mascara_c, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cs:
        pts = np.vstack([c[:, 0, :], c[:1, 0, :]])
        ax.plot(pts[:, 0], pts[:, 1], color=ACENTO, linewidth=0.55)
    sem_txt = f" · semana {reg['semana']}" if reg['semana'] else ''
    ax.set_title(f"({letra}) {nombre}{sem_txt}\n{reg['anotadas']} anotadas · {reg['C_area_calibrada']} estimadas",
                 fontsize=7.5, linespacing=1.2)
    info_47[nombre] = {'archivo': reg['archivo'], 'semana': int(reg['semana']), 'galpon': reg['galpon'],
                       'anotadas': int(reg['anotadas']), 'A_componentes': int(reg['A_componentes']),
                       'B_area_ref': int(reg['B_area_ref']), 'C_area_calibrada': int(reg['C_area_calibrada']),
                       'pct_primer_plano': round(float(reg['pct_primer_plano']), 1), 'umbral_otsu': int(reg['umbral_otsu'])}
guardar(fig, DIR_FIG4 / 'Fig_4_7_conteo_casos.png')
plt.show()
VALORES['figura_4_7'] = info_47
print(json.dumps(info_47, ensure_ascii=False, indent=1))

## 9. Guardar los valores para el texto

In [ ]:
VALORES['figuras_generadas'] = [Path(f).name for f in FIGURAS]
ruta_json = DIR_VAL / 'valores_cap04.json'
with open(ruta_json, 'w', encoding='utf-8') as fh:
    json.dump(VALORES, fh, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

print('Figuras generadas:', len(FIGURAS))
for f in FIGURAS:
    print('  ', Path(f).relative_to(DIR_LIBRO))
print('\nValores guardados en:', ruta_json.relative_to(DIR_LIBRO))
print('\nSiguiente paso: ejecute NB_LIBRO_00_Empaquetar_resultados_v2 y suba el zip al chat.')